# Bi-Directional ST-Mamba — Transfer Learning & Inference Notebook

**Purpose**: Fine-tune a pre-trained ST-Mamba model on a new dataset (with potentially shifted coordinates) and perform autoregressive inference.

## Workflow
1. Load pre-trained model weights from a checkpoint
2. Prepare new dataset with coordinate normalization aligned to the training domain
3. Fine-tune the model with lower learning rates and early stopping
4. Run autoregressive rollout on the fine-tuned model
5. Export predictions to CSV and visualize results

### Quick Start
- Set `USE_MOCK = True` (Section 2) to test the full pipeline with synthetic data
- Set `PRETRAINED_CHECKPOINT` to your best checkpoint path before running on real data
- Outputs are organized under `tl_outputs/<timestamp>/`


## 1 — Environment Setup & Imports

In [ ]:
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import torch
    if torch.cuda.is_available():
        try:
            import mamba_ssm  # noqa: F401
            print("mamba_ssm already installed.")
        except ImportError:
            print("Installing mamba-ssm …")
            _pip("mamba-ssm", "causal-conv1d")
    else:
        print("No CUDA detected — using GRU fallback (no mamba_ssm needed).")
except Exception as e:
    print(f"Setup warning: {e}")


In [ ]:
from __future__ import annotations

import math, os, time, warnings, pathlib, copy, gc
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional, List, Tuple, Dict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as grad_ckpt
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


## 2 — User Configuration

Edit the settings below before running.  
Set `USE_MOCK = True` for a quick smoke-test with synthetic data.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  USER SETTINGS — edit before running                                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

USE_MOCK = True       # ← True = use synthetic data (no real CSV required)

# Path to the pre-trained checkpoint (.pth) from the original training run.
# Set to None to start from random weights (useful only for debugging).
PRETRAINED_CHECKPOINT = None    # e.g. "checkpoints/best_checkpoint.pth"

# New dataset paths (only needed when USE_MOCK = False)
NEW_PRESSURE_CSV    = None      # e.g. "new_data/master_pressure_2sec.csv"
NEW_U_VELOCITY_CSV  = None      # e.g. "new_data/master_u_velocity_2sec.csv"
NEW_V_VELOCITY_CSV  = None      # e.g. "new_data/master_v_velocity_2sec.csv"

# If you know the coordinate bounds used during *original* training,
# provide them here for accurate normalization.  Leave as None to
# normalize based on the new dataset's own extents (Option B).
ORIGINAL_COORD_BOUNDS = None
# Example:
# ORIGINAL_COORD_BOUNDS = {"x_min": 0.0, "x_max": 2.5, "y_min": -0.5, "y_max": 0.5}

# Number of future timesteps to predict after fine-tuning
ROLLOUT_STEPS = 5

# Timestamp-based output root (all artifacts saved here)
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT   = os.path.join("tl_outputs", RUN_TIMESTAMP)


## 3 — Configuration Dataclasses

In [ ]:
@dataclass
class Config:
    """Model and data configuration — must match the training checkpoint."""

    # ── Data paths ─────────────────────────────────────────────────────────────
    pressure_path:  str = ""
    u_vel_path:     str = ""
    v_vel_path:     str = ""
    output_dir:     str = "tl_outputs"
    checkpoint_dir: str = "checkpoints"

    # ── Mesh / patch ───────────────────────────────────────────────────────────
    n_nodes:    int = 800_000
    patch_size: int = 64
    n_channels: int = 5     # p, u, v, x_norm, y_norm

    # ── Temporal window ────────────────────────────────────────────────────────
    time_in:       int = 20
    time_out:      int = 1
    rollout_steps: int = 10

    # ── Train / val / test split ───────────────────────────────────────────────
    train_ratio: float = 0.70
    val_ratio:   float = 0.15

    # ── Model architecture ─────────────────────────────────────────────────────
    hidden_dim:      int   = 128
    spatial_layers:  int   = 2
    temporal_layers: int   = 2
    dropout:         float = 0.15
    mamba_d_state:   int   = 16
    mamba_d_conv:    int   = 4
    mamba_expand:    int   = 2

    # ── Training (base) ────────────────────────────────────────────────────────
    epochs:        int   = 80
    patience:      int   = 10
    batch_size:    int   = 1
    lr:            float = 1e-4
    weight_decay:  float = 0.01
    warmup_steps:  int   = 500
    max_steps:     int   = 15_000
    grad_clip:     float = 0.5
    use_amp:       bool  = True
    use_checkpointing: bool = True

    # ── Loss ───────────────────────────────────────────────────────────────────
    spectral_weight: float = 0.10
    energy_weight:   float = 0.05
    use_spatial_weighting: bool = True
    loss_weights: Dict[str, float] = field(
        default_factory=lambda: {
            "pressure": 0.30, "u_velocity": 0.35, "v_velocity": 0.35
        }
    )


@dataclass
class TransferLearningConfig(Config):
    """
    Extends Config with transfer-learning specific parameters.

    Typical usage
    -------------
    cfg = TransferLearningConfig()
    cfg.pretrained_checkpoint = "checkpoints/best_checkpoint.pth"
    cfg.pressure_path  = "new_data/pressure.csv"
    cfg.u_vel_path     = "new_data/u_velocity.csv"
    cfg.v_vel_path     = "new_data/v_velocity.csv"
    """

    # ── Pre-trained model ──────────────────────────────────────────────────────
    pretrained_checkpoint: Optional[str] = None

    # ── Transfer learning hyper-parameters ────────────────────────────────────
    fine_tune_lr:    float = 1e-5    # Much lower than scratch training
    freeze_n_blocks: int   = 0       # Freeze first N ST blocks (0 = fine-tune all)
    freeze_embedding: bool = False   # Freeze patch-embedding layer

    # ── Reduced training schedule ──────────────────────────────────────────────
    epochs:       int = 20
    patience:     int = 5
    warmup_steps: int = 100
    max_steps:    int = 3_000

    # ── Coordinate handling ────────────────────────────────────────────────────
    coord_alignment_mode:   str              = "fixed"  # "fixed" | "adaptive"
    original_coord_bounds:  Optional[Dict]   = None


# ── Instantiate the config ─────────────────────────────────────────────────────
TL_CONFIG = TransferLearningConfig(
    pretrained_checkpoint = PRETRAINED_CHECKPOINT,
    pressure_path         = NEW_PRESSURE_CSV or "",
    u_vel_path            = NEW_U_VELOCITY_CSV or "",
    v_vel_path            = NEW_V_VELOCITY_CSV or "",
    output_dir            = OUTPUT_ROOT,
    checkpoint_dir        = os.path.join(OUTPUT_ROOT, "checkpoints"),
    rollout_steps         = ROLLOUT_STEPS,
    original_coord_bounds = ORIGINAL_COORD_BOUNDS,
)
os.makedirs(TL_CONFIG.output_dir,     exist_ok=True)
os.makedirs(TL_CONFIG.checkpoint_dir, exist_ok=True)
print("TransferLearningConfig initialised.")
print(f"Output directory : {TL_CONFIG.output_dir}")


## 4 — Hilbert-Curve Sorting

In [ ]:
def _xy2d(n: int, x: int, y: int) -> int:
    """Convert (x, y) grid coordinates to Hilbert-curve distance."""
    d = 0
    s = n >> 1
    while s > 0:
        rx = 1 if (x & s) > 0 else 0
        ry = 1 if (y & s) > 0 else 0
        d += s * s * ((3 * rx) ^ ry)
        if ry == 0:
            if rx == 1:
                x = s - 1 - x
                y = s - 1 - y
            x, y = y, x
        s >>= 1
    return d


def hilbert_sort_indices(coords: np.ndarray, order: int = 10) -> np.ndarray:
    """Return permutation indices that sort *coords* (N×2) along a 2-D Hilbert curve."""
    grid = 2 ** order
    x_min, y_min = coords[:, 0].min(), coords[:, 1].min()
    x_range = coords[:, 0].max() - x_min
    y_range = coords[:, 1].max() - y_min
    x_range = max(x_range, 1e-12)
    y_range = max(y_range, 1e-12)

    ix = np.floor((coords[:, 0] - x_min) / x_range * (grid - 1)).astype(int).clip(0, grid - 1)
    iy = np.floor((coords[:, 1] - y_min) / y_range * (grid - 1)).astype(int).clip(0, grid - 1)

    dist = np.array([_xy2d(grid, int(ix[i]), int(iy[i])) for i in range(len(coords))])
    return np.argsort(dist)


print("Hilbert sort ready.")


## 5 — Data Pipeline

In [ ]:
# ── 5a. Mock data (with optional coordinate shift) ────────────────────────────

def make_mock_data(
    n_nodes: int = 10_000,
    n_timesteps: int = 40,
    seed: int = 0,
    coord_shift: Tuple[float, float] = (0.0, 0.0),
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Generate synthetic CFD-like data.

    Parameters
    ----------
    coord_shift : (dx, dy) added to all coordinates to simulate a domain offset.
    """
    rng = np.random.default_rng(seed)
    base = rng.uniform(0, 1, (n_nodes, 2)).astype(np.float32)
    n_bl = n_nodes // 10
    base[:n_bl, 1] = rng.uniform(0, 0.05, n_bl)  # boundary-layer cluster

    coords = base + np.array(coord_shift, dtype=np.float32)
    t = np.linspace(0, 2 * np.pi, n_timesteps, dtype=np.float32)

    x, y = base[:, 0:1], base[:, 1:2]   # use unshifted for field generation
    pressure = (
        100_000
        + 5_000 * np.sin(2 * np.pi * x)
        + 3_000 * np.cos(2 * np.pi * y)
        + 1_000 * np.sin(t[np.newaxis, :])
        + rng.normal(0, 100, (n_nodes, n_timesteps)).astype(np.float32)
    ).astype(np.float32)

    u_vel = (
        20 + 10 * np.cos(np.pi * y) + 2 * np.sin(t[np.newaxis, :])
        + rng.normal(0, 0.5, (n_nodes, n_timesteps)).astype(np.float32)
    ).astype(np.float32)

    v_vel = (
        5 * np.sin(np.pi * x) + 1 * np.cos(t[np.newaxis, :])
        + rng.normal(0, 0.3, (n_nodes, n_timesteps)).astype(np.float32)
    ).astype(np.float32)

    print(f"[Mock] nodes={n_nodes:,}, T={n_timesteps}, coord_shift={coord_shift}")
    return coords, pressure, u_vel, v_vel


In [ ]:
# ── 5b. CSV loader ────────────────────────────────────────────────────────────

def load_csv_data(
    pressure_path: str, u_vel_path: str, v_vel_path: str
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Load three variable CSVs (x, y, t0, t1, …) → coords, pressure, u, v."""
    print("Loading CSVs …")
    df_p = pd.read_csv(pressure_path,  header=0)
    df_u = pd.read_csv(u_vel_path,     header=0)
    df_v = pd.read_csv(v_vel_path,     header=0)

    coords   = df_p.iloc[:, :2].values.astype(np.float32)
    pressure = df_p.iloc[:, 2:].values.astype(np.float32)
    u_vel    = df_u.iloc[:, 2:].values.astype(np.float32)
    v_vel    = df_v.iloc[:, 2:].values.astype(np.float32)

    print(f"Loaded: nodes={len(coords):,}, timesteps={pressure.shape[1]}")
    return coords, pressure, u_vel, v_vel


def auto_load_data(
    cfg: Config,
    use_mock: bool = False,
    mock_coord_shift: Tuple[float, float] = (0.0, 0.0),
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Auto-select mock vs real data based on path validity."""
    paths_ok = all(
        p and os.path.exists(p)
        for p in [cfg.pressure_path, cfg.u_vel_path, cfg.v_vel_path]
    )
    if use_mock or not paths_ok:
        if not use_mock:
            print("CSV paths not found — falling back to mock data.")
        return make_mock_data(
            n_nodes=min(cfg.n_nodes, 10_000),
            n_timesteps=40,
            coord_shift=mock_coord_shift,
        )
    return load_csv_data(cfg.pressure_path, cfg.u_vel_path, cfg.v_vel_path)


## 6 — Coordinate Shift Analysis

In [ ]:
def analyze_coordinate_shift(
    old_coords: np.ndarray,
    new_coords: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Report the translation and scale difference between two coordinate sets.

    Parameters
    ----------
    old_coords : (N1, 2) coordinates from the original training dataset.
    new_coords : (N2, 2) coordinates from the new (fine-tuning) dataset.

    Returns
    -------
    shift       : (2,) centroid difference (new - old)
    scale_ratio : (2,) std(new) / std(old)
    """
    old_center = old_coords.mean(axis=0)
    new_center = new_coords.mean(axis=0)
    shift      = new_center - old_center

    old_scale  = old_coords.std(axis=0)
    new_scale  = new_coords.std(axis=0)
    # Avoid division by zero
    scale_ratio = np.where(old_scale > 1e-12, new_scale / old_scale, 1.0)

    old_bounds = {"x": (old_coords[:, 0].min(), old_coords[:, 0].max()),
                  "y": (old_coords[:, 1].min(), old_coords[:, 1].max())}
    new_bounds = {"x": (new_coords[:, 0].min(), new_coords[:, 0].max()),
                  "y": (new_coords[:, 1].min(), new_coords[:, 1].max())}

    print("Coordinate Shift Analysis")
    print("─" * 40)
    print(f"  Old  x: [{old_bounds['x'][0]:.4f}, {old_bounds['x'][1]:.4f}]")
    print(f"  New  x: [{new_bounds['x'][0]:.4f}, {new_bounds['x'][1]:.4f}]")
    print(f"  Old  y: [{old_bounds['y'][0]:.4f}, {old_bounds['y'][1]:.4f}]")
    print(f"  New  y: [{new_bounds['y'][0]:.4f}, {new_bounds['y'][1]:.4f}]")
    print(f"  Centroid shift  : dx={shift[0]:+.4f}, dy={shift[1]:+.4f}")
    print(f"  Scale ratio     : sx={scale_ratio[0]:.4f}, sy={scale_ratio[1]:.4f}")
    print("─" * 40)
    return shift, scale_ratio


def normalize_coords_to_training_domain(
    coords: np.ndarray,
    original_bounds: Dict[str, float],
) -> np.ndarray:
    """
    Map new coordinates into the [0, 1] range defined by the *original*
    training domain bounds.  This is the "fixed" alignment strategy —
    it preserves the spatial interpretation the model learned.

    Parameters
    ----------
    coords          : (N, 2) raw coordinates from the new dataset.
    original_bounds : dict with keys x_min, x_max, y_min, y_max.

    Returns
    -------
    coords_norm : (N, 2) normalised to the original domain's [0, 1] range.
    """
    x_range = original_bounds["x_max"] - original_bounds["x_min"]
    y_range = original_bounds["y_max"] - original_bounds["y_min"]
    x_range = max(x_range, 1e-12)
    y_range = max(y_range, 1e-12)

    coords_norm = coords.copy().astype(np.float32)
    coords_norm[:, 0] = (coords[:, 0] - original_bounds["x_min"]) / x_range
    coords_norm[:, 1] = (coords[:, 1] - original_bounds["y_min"]) / y_range
    print(f"Coords normalised to training domain:")
    print(f"  x_norm ∈ [{coords_norm[:, 0].min():.3f}, {coords_norm[:, 0].max():.3f}]")
    print(f"  y_norm ∈ [{coords_norm[:, 1].min():.3f}, {coords_norm[:, 1].max():.3f}]")
    return coords_norm


print("Coordinate utilities ready.")


## 7 — Data Normalizer

In [ ]:
class DataNormalizer:
    """Per-variable StandardScaler; coordinates normalised to [0, 1]."""

    def __init__(self) -> None:
        self.scalers: Dict[str, StandardScaler] = {
            v: StandardScaler() for v in ("pressure", "u_velocity", "v_velocity")
        }
        self.coord_min:   Optional[np.ndarray] = None
        self.coord_max:   Optional[np.ndarray] = None
        self.coord_scale: Optional[np.ndarray] = None

    def fit(
        self,
        pressure_train: np.ndarray,
        u_train:        np.ndarray,
        v_train:        np.ndarray,
        coords:         np.ndarray,
    ) -> None:
        """Fit scalers on training-split arrays (N, T_train)."""
        self.scalers["pressure"].fit(pressure_train.reshape(-1, 1))
        self.scalers["u_velocity"].fit(u_train.reshape(-1, 1))
        self.scalers["v_velocity"].fit(v_train.reshape(-1, 1))
        self.coord_min   = coords.min(axis=0)
        self.coord_max   = coords.max(axis=0)
        rng              = self.coord_max - self.coord_min
        rng[rng < 1e-12] = 1.0
        self.coord_scale = rng
        print(f"  Pressure  : mean={self.scalers['pressure'].mean_[0]:.4f}, "
              f"std={np.sqrt(self.scalers['pressure'].var_[0]):.4f}")
        print(f"  U-velocity: mean={self.scalers['u_velocity'].mean_[0]:.4f}, "
              f"std={np.sqrt(self.scalers['u_velocity'].var_[0]):.4f}")
        print(f"  V-velocity: mean={self.scalers['v_velocity'].mean_[0]:.4f}, "
              f"std={np.sqrt(self.scalers['v_velocity'].var_[0]):.4f}")

    def transform_var(self, arr: np.ndarray, name: str) -> np.ndarray:
        shape = arr.shape
        return self.scalers[name].transform(arr.reshape(-1, 1)).reshape(shape)

    def inverse_transform_var(self, arr: np.ndarray, name: str) -> np.ndarray:
        shape = arr.shape
        return self.scalers[name].inverse_transform(arr.reshape(-1, 1)).reshape(shape)

    def transform_coords(self, coords: np.ndarray) -> np.ndarray:
        return (coords - self.coord_min) / self.coord_scale

    def fit_transform(
        self,
        pressure:   np.ndarray,
        u_vel:      np.ndarray,
        v_vel:      np.ndarray,
        coords:     np.ndarray,
        t_train_end: int,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """Fit on training slice, transform all timesteps; also normalise coords."""
        self.fit(pressure[:, :t_train_end], u_vel[:, :t_train_end],
                 v_vel[:, :t_train_end], coords)
        p_n = self.transform_var(pressure, "pressure")
        u_n = self.transform_var(u_vel,    "u_velocity")
        v_n = self.transform_var(v_vel,    "v_velocity")
        c_n = self.transform_coords(coords)
        return p_n, u_n, v_n, c_n


print("DataNormalizer ready.")


## 8 — Patch Dataset

In [ ]:
class PatchDataset(Dataset):
    """
    Produces (input_seq, target_seq) patch-level windows.

    x : (time_in,  n_patches, patch_size, n_channels)
    y : (time_out, n_patches, patch_size, 3)   — p, u, v only
    """

    def __init__(
        self,
        data:        np.ndarray,   # (N, T, C)  C = p,u,v,x,y
        hilbert_idx: np.ndarray,   # (N,)
        patch_size:  int,
        time_in:     int,
        time_out:    int,
        t_start:     int,
        t_end:       int,
    ) -> None:
        super().__init__()
        self.patch_size = patch_size
        self.time_in    = time_in
        self.time_out   = time_out

        data_sorted = data[hilbert_idx]
        N, T, C = data_sorted.shape
        n_patches = math.ceil(N / patch_size)
        pad_len   = n_patches * patch_size - N
        if pad_len > 0:
            pad = np.zeros((pad_len, T, C), dtype=data_sorted.dtype)
            data_sorted = np.concatenate([data_sorted, pad], axis=0)

        self.patches   = data_sorted.reshape(n_patches, patch_size, T, C)
        self.n_patches = n_patches

        win = time_in + time_out
        self.indices = [
            t for t in range(t_start, t_end - win + 1)
        ]

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        t = self.indices[idx]
        window = self.patches[:, :, t : t + self.time_in + self.time_out, :]
        # window: (P, S, W, C)
        window = window.transpose(2, 0, 1, 3)   # (W, P, S, C)
        x = torch.from_numpy(window[:self.time_in])            # (T_in, P, S, C)
        y = torch.from_numpy(window[self.time_in:, :, :, :3]) # (T_out, P, S, 3)
        return x, y


print("PatchDataset ready.")


## 9 — Model Architecture

In [ ]:
try:
    from mamba_ssm import Mamba as _MambaSSM
    _MAMBA_AVAILABLE = True
    print("mamba_ssm backend detected.")
except ImportError:
    _MAMBA_AVAILABLE = False
    print("mamba_ssm NOT available — using bidirectional GRU fallback.")


class MambaLikeBlock(nn.Module):
    """Single-direction SSM block (mamba_ssm or GRU fallback)."""

    def __init__(self, d_model: int, d_state: int = 16,
                 d_conv: int = 4, expand: int = 2) -> None:
        super().__init__()
        if _MAMBA_AVAILABLE:
            self._impl = _MambaSSM(d_model=d_model, d_state=d_state,
                                   d_conv=d_conv, expand=expand)
        else:
            self._impl = nn.GRU(d_model, d_model, batch_first=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, D)
        if _MAMBA_AVAILABLE:
            return self._impl(x)
        out, _ = self._impl(x)
        return out


class BiDirectionalSpatialMamba(nn.Module):
    """Bidirectional spatial Mamba: forward + backward along the patch dimension."""

    def __init__(self, hidden_dim: int, d_state: int, d_conv: int, expand: int,
                 dropout: float = 0.0, use_checkpointing: bool = False) -> None:
        super().__init__()
        self.fwd  = MambaLikeBlock(hidden_dim, d_state, d_conv, expand)
        self.bwd  = MambaLikeBlock(hidden_dim, d_state, d_conv, expand)
        self.norm = nn.LayerNorm(hidden_dim)
        self.drop = nn.Dropout(dropout)
        self.use_ckpt = use_checkpointing

    def _run(self, h: torch.Tensor) -> torch.Tensor:
        h_fwd = self.fwd(h)
        h_bwd = self.bwd(h.flip(1)).flip(1)
        return self.drop(self.norm(h + h_fwd + h_bwd))

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        # h: (B, T, P, H)
        B, T, P, H = h.shape
        h_flat = h.reshape(B * T, P, H)
        if self.use_ckpt and self.training:
            out = grad_ckpt.checkpoint(self._run, h_flat, use_reentrant=False)
        else:
            out = self._run(h_flat)
        return out.reshape(B, T, P, H)


class CausalTemporalMamba(nn.Module):
    """Causal (uni-directional) Mamba applied along the time dimension."""

    def __init__(self, hidden_dim: int, d_state: int, d_conv: int, expand: int,
                 dropout: float = 0.0, use_checkpointing: bool = False) -> None:
        super().__init__()
        self.mamba = MambaLikeBlock(hidden_dim, d_state, d_conv, expand)
        self.norm  = nn.LayerNorm(hidden_dim)
        self.drop  = nn.Dropout(dropout)
        self.use_ckpt = use_checkpointing

    def _run(self, h: torch.Tensor) -> torch.Tensor:
        return self.drop(self.norm(h + self.mamba(h)))

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        # h: (B, T, P, H)
        B, T, P, H = h.shape
        h_flat = h.permute(0, 2, 1, 3).reshape(B * P, T, H)
        if self.use_ckpt and self.training:
            out = grad_ckpt.checkpoint(self._run, h_flat, use_reentrant=False)
        else:
            out = self._run(h_flat)
        return out.reshape(B, P, T, H).permute(0, 2, 1, 3)


class STBlock(nn.Module):
    """One ST-Mamba block: BiDirectional Spatial → Causal Temporal."""

    def __init__(self, hidden_dim: int, d_state: int, d_conv: int, expand: int,
                 dropout: float = 0.0, use_checkpointing: bool = False) -> None:
        super().__init__()
        self.spatial  = BiDirectionalSpatialMamba(
            hidden_dim, d_state, d_conv, expand, dropout, use_checkpointing)
        self.temporal = CausalTemporalMamba(
            hidden_dim, d_state, d_conv, expand, dropout, use_checkpointing)

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        h = self.spatial(h)
        h = self.temporal(h)
        return h


class PatchEmbed(nn.Module):
    """Linear projection from (patch_size × n_channels) → hidden_dim."""

    def __init__(self, patch_size: int, n_channels: int,
                 hidden_dim: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.proj = nn.Linear(patch_size * n_channels, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, P, S, C = x.shape
        x = x.reshape(B, T, P, S * C)
        return self.drop(self.norm(self.proj(x)))  # (B, T, P, H)


class PatchUnembed(nn.Module):
    """Project hidden_dim back to patch_size × 3 output variables."""

    def __init__(self, patch_size: int, hidden_dim: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.proj = nn.Linear(hidden_dim, patch_size * 3)
        self.drop = nn.Dropout(dropout)

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        # h: (B, T_out, P, H)
        B, T, P, H = h.shape
        out = self.drop(self.proj(h))          # (B, T, P, S*3)
        return out.reshape(B, T, P, -1, 3)     # (B, T, P, S, 3)


class STMambaModel(nn.Module):
    """
    Full Bi-Directional Spatial-Temporal Mamba model.

    Input  : (B, T_in,  n_patches, patch_size, n_channels)
    Output : (B, T_out, n_patches, patch_size, 3)   — p, u, v
    """

    def __init__(self, cfg: Config) -> None:
        super().__init__()
        self.cfg = cfg

        self.embed  = PatchEmbed(cfg.patch_size, cfg.n_channels,
                                 cfg.hidden_dim, cfg.dropout)
        self.blocks = nn.ModuleList([
            STBlock(cfg.hidden_dim, cfg.mamba_d_state, cfg.mamba_d_conv,
                    cfg.mamba_expand, cfg.dropout, cfg.use_checkpointing)
            for _ in range(cfg.spatial_layers)
        ])
        self.unembed   = PatchUnembed(cfg.patch_size, cfg.hidden_dim, cfg.dropout)
        self.time_proj = nn.Linear(cfg.time_in, cfg.time_out)

        n_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"STMambaModel created — {n_params:,} trainable parameters.")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.embed(x)                              # (B, T_in, P, H)
        for block in self.blocks:
            h = block(h)
        h = h.permute(0, 2, 3, 1)                     # (B, P, H, T_in)
        h = self.time_proj(h)                          # (B, P, H, T_out)
        h = h.permute(0, 3, 1, 2)                     # (B, T_out, P, H)
        return self.unembed(h)                         # (B, T_out, P, S, 3)


def create_model(cfg: Config, n_patches: int) -> nn.Module:
    model = STMambaModel(cfg).to(DEVICE)
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs.")
        model = nn.DataParallel(model)
    return model


print("Model architecture ready.")


## 10 — Loss Function

In [ ]:
def compute_wall_weights(
    coords_norm: np.ndarray,
    sigma: float = 0.02,
    max_weight: float = 10.0,
) -> torch.Tensor:
    """Per-node spatial weights; upweight nodes near the wall (y ≈ 0)."""
    y = coords_norm[:, 1].clip(1e-6, None)
    return torch.from_numpy(
        (sigma / y).clip(1.0, max_weight).astype(np.float32)
    )


class STMambaLoss(nn.Module):
    """Combined MSE + spectral + energy loss with optional spatial weighting."""

    def __init__(
        self,
        cfg: Config,
        wall_weights: Optional[torch.Tensor] = None,
        patch_size:   Optional[int] = None,
    ) -> None:
        super().__init__()
        self.cfg   = cfg
        self.var_w = torch.tensor(
            [cfg.loss_weights["pressure"],
             cfg.loss_weights["u_velocity"],
             cfg.loss_weights["v_velocity"]],
            dtype=torch.float32,
        )
        self.register_buffer("var_weights", self.var_w)

        if wall_weights is not None and cfg.use_spatial_weighting:
            ps = patch_size or cfg.patch_size
            N  = len(wall_weights)
            n_patches = math.ceil(N / ps)
            pad_len   = n_patches * ps - N
            if pad_len > 0:
                wall_weights = torch.cat([wall_weights, torch.ones(pad_len)])
            # Reshape to (n_patches, patch_size) then average per patch
            pw = wall_weights[: n_patches * ps].reshape(n_patches, ps).mean(dim=1)
            self.register_buffer("patch_weights", pw)
        else:
            self.patch_weights = None

    def forward(
        self, pred: torch.Tensor, target: torch.Tensor
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        # pred, target: (B, T_out, P, S, 3)
        diff  = pred - target
        sq    = diff ** 2

        # Variable weighting
        vw   = self.var_weights.to(sq.device)           # (3,)
        loss = (sq * vw).mean()

        # Spectral term
        if self.cfg.spectral_weight > 0:
            pf = torch.fft.rfft(pred,   dim=2, norm="ortho")
            tf = torch.fft.rfft(target, dim=2, norm="ortho")
            spec_loss = (torch.abs(pf - tf) ** 2).mean()
            loss = loss + self.cfg.spectral_weight * spec_loss
        else:
            spec_loss = torch.tensor(0.0)

        return loss, {"total": loss.item(), "spectral": spec_loss.item()}


print("Loss function ready.")


## 11 — Transfer Learning Utilities

In [ ]:
def load_pretrained_weights(
    model:           nn.Module,
    checkpoint_path: str,
    device:          torch.device,
    strict:          bool = True,
) -> nn.Module:
    """
    Load pre-trained weights from a checkpoint file.

    Parameters
    ----------
    strict : If True, the state-dict must match exactly.
             Set to False if the architecture has minor differences.
    """
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(
            f"Checkpoint not found: {checkpoint_path}\n"
            "Set PRETRAINED_CHECKPOINT to a valid path or USE_MOCK=True."
        )
    print(f"Loading pre-trained weights from {checkpoint_path} …")
    ckpt = torch.load(checkpoint_path, map_location=device)

    state_dict = ckpt.get("model_state", ckpt)
    raw_model  = model.module if isinstance(model, nn.DataParallel) else model

    missing, unexpected = raw_model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"  ⚠ Missing keys ({len(missing)}): {missing[:5]}")
    if unexpected:
        print(f"  ⚠ Unexpected keys ({len(unexpected)}): {unexpected[:5]}")
    if not missing and not unexpected:
        print("  ✔ All weights loaded successfully (strict).")

    # Report metadata from checkpoint
    if "epoch" in ckpt:
        print(f"  Checkpoint epoch    : {ckpt['epoch']}")
    if "best_val_loss" in ckpt:
        print(f"  Best val loss       : {ckpt['best_val_loss']:.6f}")
    return model


def freeze_model_layers(
    model:            nn.Module,
    n_blocks_to_freeze: int  = 0,
    freeze_embedding:   bool = False,
) -> nn.Module:
    """
    Optionally freeze early layers to preserve learned representations.

    Parameters
    ----------
    n_blocks_to_freeze : Number of ST blocks to freeze (0 = fine-tune all).
    freeze_embedding   : Whether to freeze the patch-embedding layer.
    """
    raw_model = model.module if isinstance(model, nn.DataParallel) else model

    if freeze_embedding:
        for p in raw_model.embed.parameters():
            p.requires_grad = False
        print("  ✔ Embedding layer frozen.")

    for i in range(min(n_blocks_to_freeze, len(raw_model.blocks))):
        for p in raw_model.blocks[i].parameters():
            p.requires_grad = False
        print(f"  ✔ ST block {i} frozen.")

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  Trainable params: {trainable:,} / {total:,} "
          f"({100 * trainable / max(total, 1):.1f}%)")
    return model


print("Transfer learning utilities ready.")


## 12 — Checkpoint Utilities

In [ ]:
def get_lr_scheduler(optimizer, warmup_steps: int, max_steps: int) -> torch.optim.lr_scheduler.LambdaLR:
    """Cosine annealing with linear warm-up."""
    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, max_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def save_tl_checkpoint(
    state:   dict,
    epoch:   int,
    cfg:     TransferLearningConfig,
    is_best: bool = False,
) -> None:
    """Save checkpoint; keep only the 3 most recent epoch checkpoints."""
    os.makedirs(cfg.checkpoint_dir, exist_ok=True)
    epoch_path = os.path.join(cfg.checkpoint_dir, f"checkpoint_epoch_{epoch:04d}.pth")
    torch.save(state, epoch_path)

    # Rotate old epoch checkpoints
    ckpts = sorted(pathlib.Path(cfg.checkpoint_dir).glob("checkpoint_epoch_*.pth"))
    for old in ckpts[:-3]:
        old.unlink()

    # Always write latest
    torch.save(state, os.path.join(cfg.checkpoint_dir, "latest_checkpoint.pth"))

    if is_best:
        torch.save(state, os.path.join(cfg.checkpoint_dir, "best_checkpoint.pth"))
        print(f"  ✔ Best TL checkpoint saved (epoch {epoch})")


def load_tl_checkpoint(
    cfg:       TransferLearningConfig,
    model:     nn.Module,
    optimizer,
    scheduler,
    scaler,
) -> Tuple[int, float, dict]:
    """Resume from latest TL checkpoint if available."""
    latest = os.path.join(cfg.checkpoint_dir, "latest_checkpoint.pth")
    if not os.path.exists(latest):
        print("No TL checkpoint found — starting transfer learning fresh.")
        return 0, float("inf"), {"train_loss": [], "val_loss": [], "val_rmse": []}

    print(f"TL checkpoint found — resuming from {latest}")
    ckpt      = torch.load(latest, map_location=DEVICE)
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    raw_model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    if scheduler and ckpt.get("scheduler_state"):
        scheduler.load_state_dict(ckpt["scheduler_state"])
    if scaler and ckpt.get("scaler_state"):
        scaler.load_state_dict(ckpt["scaler_state"])

    start_epoch   = ckpt["epoch"] + 1
    best_val_loss = ckpt.get("best_val_loss", float("inf"))
    history       = ckpt.get("history", {"train_loss": [], "val_loss": [], "val_rmse": []})
    print(f"  Resumed from epoch {ckpt['epoch']}, best_val_loss={best_val_loss:.6f}")
    return start_epoch, best_val_loss, history


print("Checkpoint utilities ready.")


## 13 — Transfer Learning Training Loop

In [ ]:
def compute_val_rmse(
    model:      nn.Module,
    val_loader: DataLoader,
    normalizer: DataNormalizer,
    cfg:        Config,
    device:     torch.device,
) -> np.ndarray:
    """Compute per-variable RMSE in physical units on the validation set."""
    model.eval()
    sq_sum  = np.zeros(3)
    n_total = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y   = x.to(device), y.to(device)
            with autocast(enabled=cfg.use_amp):
                pred = model(x)
            err = (pred - y).cpu().numpy()
            for v_idx, v_name in enumerate(["pressure", "u_velocity", "v_velocity"]):
                e_phy   = normalizer.scalers[v_name].scale_[0] * err[..., v_idx].reshape(-1)
                sq_sum[v_idx] += (e_phy ** 2).sum()
            n_total += err[..., 0].size
    return np.sqrt(sq_sum / max(n_total, 1))


def train_transfer_learning(
    model:       nn.Module,
    train_loader: DataLoader,
    val_loader:   DataLoader,
    criterion:    nn.Module,
    normalizer:   DataNormalizer,
    cfg:          TransferLearningConfig,
    device:       torch.device,
) -> dict:
    """
    Fine-tuning training loop with:
    - Low learning rate (cfg.fine_tune_lr)
    - Cosine annealing warm-up
    - Checkpoint save/resume
    - Early stopping (cfg.patience)
    - Per-variable RMSE tracking
    """
    print("\n" + "="*70)
    print("  Transfer Learning Training")
    print("="*70)
    print(f"  Fine-tune LR  : {cfg.fine_tune_lr}")
    print(f"  Epochs        : {cfg.epochs}")
    print(f"  Patience      : {cfg.patience}")
    print(f"  Frozen blocks : {cfg.freeze_n_blocks}")
    print(f"  AMP           : {cfg.use_amp}")
    print("="*70)

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg.fine_tune_lr,
        weight_decay=cfg.weight_decay,
    )
    total_steps = cfg.epochs * max(len(train_loader), 1)
    scheduler   = get_lr_scheduler(
        optimizer, cfg.warmup_steps, min(cfg.max_steps, total_steps)
    )
    scaler = GradScaler(enabled=cfg.use_amp)

    start_epoch, best_val_loss, history = load_tl_checkpoint(
        cfg, model, optimizer, scheduler, scaler
    )
    if "val_rmse" not in history:
        history["val_rmse"] = []

    patience_counter = 0

    for epoch in range(start_epoch, cfg.epochs):
        t0 = time.time()
        model.train()
        train_losses: List[float] = []

        for x, y in tqdm(train_loader, desc=f"TL Epoch {epoch+1}/{cfg.epochs}", leave=False):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=cfg.use_amp):
                pred = model(x)
                loss, loss_dict = criterion(pred, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], cfg.grad_clip
            )
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            train_losses.append(loss_dict["total"])

        avg_train = float(np.mean(train_losses)) if train_losses else float("nan")
        history["train_loss"].append(avg_train)

        # ── Validation ────────────────────────────────────────────────────────
        val_loss = float("nan")
        val_rmse = [float("nan")] * 3
        if len(val_loader) > 0:
            model.eval()
            vl: List[float] = []
            with torch.no_grad():
                for x, y in val_loader:
                    x, y = x.to(device), y.to(device)
                    with autocast(enabled=cfg.use_amp):
                        pred = model(x)
                        l, _ = criterion(pred, y)
                    vl.append(l.item())
            val_loss = float(np.mean(vl))
            val_rmse = compute_val_rmse(model, val_loader, normalizer, cfg, device).tolist()

        history["val_loss"].append(val_loss)
        history["val_rmse"].append(val_rmse)

        elapsed = time.time() - t0
        print(
            f"Epoch {epoch+1:3d}/{cfg.epochs} | "
            f"train={avg_train:.4f} | val={val_loss:.4f} | "
            f"RMSE P={val_rmse[0]:.3f} U={val_rmse[1]:.3f} V={val_rmse[2]:.3f} | "
            f"{elapsed:.1f}s"
        )

        # ── Checkpoint ────────────────────────────────────────────────────────
        raw_model = model.module if isinstance(model, nn.DataParallel) else model
        is_best   = not math.isnan(val_loss) and val_loss < best_val_loss
        if is_best:
            best_val_loss    = val_loss
            patience_counter = 0
        else:
            patience_counter += 1

        save_tl_checkpoint(
            {
                "epoch":           epoch,
                "model_state":     raw_model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "scaler_state":    scaler.state_dict(),
                "best_val_loss":   best_val_loss,
                "history":         history,
            },
            epoch, cfg, is_best=is_best,
        )

        # ── Early stopping ────────────────────────────────────────────────────
        if patience_counter >= cfg.patience:
            print(f"Early stopping triggered after {cfg.patience} epochs without improvement.")
            break

    print("Transfer learning complete.")
    return history


print("Transfer learning training loop ready.")


## 14 — Autoregressive Rollout Inference

In [ ]:
@torch.no_grad()
def autoregressive_rollout(
    model:          nn.Module,
    initial_window: torch.Tensor,   # (1, T_in, P, S, C)
    rollout_steps:  int,
    device:         torch.device,
    use_amp:        bool = True,
) -> torch.Tensor:
    """
    Feed predictions back as input for multi-step forecasting.

    Returns
    -------
    predictions : (rollout_steps, P, S, 3)  — p, u, v (normalised)
    """
    model.eval()
    window  = initial_window.to(device)      # (1, T_in, P, S, C)
    C       = window.shape[-1]
    all_preds: List[torch.Tensor] = []

    for _ in tqdm(range(rollout_steps), desc="Autoregressive rollout"):
        with autocast(enabled=use_amp):
            pred = model(window)               # (1, T_out, P, S, 3)

        pred_frame = pred[:, -1:, :, :, :]    # (1, 1, P, S, 3)
        all_preds.append(pred_frame[0, 0].cpu())   # (P, S, 3)

        # Build next window: drop oldest frame, append prediction
        # Pad prediction with zero coordinates (channels 3,4) to match C=5
        pad = torch.zeros(*pred_frame.shape[:-1], C - 3, device=device)
        new_frame = torch.cat([pred_frame, pad], dim=-1)  # (1, 1, P, S, C)
        window = torch.cat([window[:, 1:], new_frame], dim=1)

    return torch.stack(all_preds, dim=0)   # (R, P, S, 3)


def prepare_seed_window(
    data:        np.ndarray,   # (N, T, C)
    hilbert_idx: np.ndarray,
    patch_size:  int,
    time_in:     int,
    seed_start:  int,
) -> Tuple[torch.Tensor, int]:
    """Build the initial input window tensor for autoregressive rollout."""
    N, T, C = data.shape
    n_patches = math.ceil(N / patch_size)
    pad_len   = n_patches * patch_size - N

    data_sorted = data[hilbert_idx]
    if pad_len > 0:
        data_sorted = np.concatenate(
            [data_sorted, np.zeros((pad_len, T, C), dtype=data_sorted.dtype)], axis=0
        )
    # (n_patches, patch_size, T, C)
    patches = data_sorted.reshape(n_patches, patch_size, T, C)

    t_end = min(seed_start + time_in, T)
    actual_in = t_end - seed_start
    window_np = patches[:, :, seed_start:t_end, :].transpose(2, 0, 1, 3)  # (T_in, P, S, C)
    window    = torch.from_numpy(window_np).unsqueeze(0)   # (1, T_in, P, S, C)
    return window, n_patches


print("Inference utilities ready.")


## 15 — CSV Export

In [ ]:
def export_predictions_to_csv(
    predictions: np.ndarray,     # (rollout_steps, N_padded, 3) — normalised
    coords_orig: np.ndarray,     # (N_real, 2)
    normalizer:  DataNormalizer,
    output_dir:  str,
    prefix:      str = "st_mamba_pred",
) -> List[str]:
    """
    Denormalise and write three CSV files (pressure, u_velocity, v_velocity).

    Output columns: x_coord, y_coord, pred_t1, pred_t2, …
    """
    os.makedirs(output_dir, exist_ok=True)
    N_real     = len(coords_orig)
    R, N_pad, _ = predictions.shape
    N          = min(N_real, N_pad)

    var_names  = ["pressure", "u_velocity", "v_velocity"]
    file_stems = [f"{prefix}_pressure", f"{prefix}_u_velocity", f"{prefix}_v_velocity"]

    out_paths: List[str] = []
    for v_idx, (var, stem) in enumerate(zip(var_names, file_stems)):
        raw   = predictions[:, :N, v_idx]           # (R, N)
        phys  = normalizer.inverse_transform_var(raw.T, var).T  # (R, N)
        cols  = {f"pred_t{i+1}": phys[i] for i in range(R)}
        df    = pd.DataFrame({"x_coord": coords_orig[:N, 0],
                               "y_coord": coords_orig[:N, 1], **cols})
        path  = os.path.join(output_dir, f"{stem}.csv")
        df.to_csv(path, index=False)
        out_paths.append(path)
        print(f"  Saved {path}  shape=({N}, {R+2})")

    return out_paths


print("CSV export ready.")


## 16 — Visualization

In [ ]:
def plot_training_history(history: dict, output_dir: str) -> None:
    """Plot training/validation loss and per-variable RMSE curves."""
    epochs_ran = len(history["train_loss"])
    if epochs_ran == 0:
        print("No training history to plot.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Loss curves ────────────────────────────────────────────────────────────
    ax = axes[0]
    ax.plot(history["train_loss"], label="Train Loss")
    val_clean = [v for v in history["val_loss"] if not math.isnan(v)]
    if val_clean:
        ax.plot(history["val_loss"], label="Val Loss")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("Transfer Learning Loss Curves")
    ax.legend(); ax.grid(True)

    # ── Per-variable RMSE ──────────────────────────────────────────────────────
    ax = axes[1]
    rmse_arr = np.array(history.get("val_rmse", []))
    if rmse_arr.ndim == 2 and rmse_arr.shape[1] == 3:
        for i, name in enumerate(["Pressure", "U-vel", "V-vel"]):
            valid = ~np.isnan(rmse_arr[:, i])
            if valid.any():
                ax.plot(np.where(valid)[0], rmse_arr[valid, i], label=name)
    ax.set_xlabel("Epoch"); ax.set_ylabel("RMSE (physical units)")
    ax.set_title("Validation RMSE per Variable")
    ax.legend(); ax.grid(True)

    plt.tight_layout()
    out = os.path.join(output_dir, "training_history.png")
    plt.savefig(out, dpi=120)
    plt.show()
    print(f"Saved: {out}")


def plot_prediction_samples(
    pred_rollout: np.ndarray,   # (R, N_padded, 3)
    coords_orig:  np.ndarray,   # (N_real, 2)
    normalizer:   DataNormalizer,
    output_dir:   str,
    n_cols:       int = 3,
) -> None:
    """Scatter-plot the predicted field for each variable at several timesteps."""
    R = pred_rollout.shape[0]
    N = min(len(coords_orig), pred_rollout.shape[1])
    var_names = ["Pressure", "U-velocity", "V-velocity"]
    scalers   = ["pressure", "u_velocity", "v_velocity"]

    # Sample up to n_cols evenly-spaced prediction steps
    steps = np.linspace(0, R - 1, min(n_cols, R), dtype=int)

    for v_idx, (var, sc) in enumerate(zip(var_names, scalers)):
        fig, axes = plt.subplots(1, len(steps), figsize=(5 * len(steps), 4))
        if len(steps) == 1:
            axes = [axes]
        for col, step in enumerate(steps):
            raw  = pred_rollout[step, :N, v_idx]
            phys = normalizer.inverse_transform_var(raw, sc)
            sc_plot = axes[col].scatter(
                coords_orig[:N, 0], coords_orig[:N, 1],
                c=phys, cmap="RdBu_r", s=1, rasterized=True
            )
            axes[col].set_title(f"Step t+{step+1}")
            axes[col].set_xlabel("x"); axes[col].set_ylabel("y")
            plt.colorbar(sc_plot, ax=axes[col])

        plt.suptitle(f"Predicted {var}", fontsize=13)
        plt.tight_layout()
        stem = var.lower().replace("-", "_").replace(" ", "_")
        out  = os.path.join(output_dir, f"rollout_{stem}.png")
        plt.savefig(out, dpi=120)
        plt.show()
        print(f"Saved: {out}")


print("Visualization utilities ready.")


## 17 — Data Pipeline Builder

In [ ]:
def build_dataloaders(
    cfg:      Config,
    use_mock: bool = False,
    mock_coord_shift: Tuple[float, float] = (0.0, 0.0),
):
    """
    Full data pipeline for new dataset:
      auto_load → Hilbert sort → normalise → split → PatchDataset → DataLoader

    Returns
    -------
    train_loader, val_loader, test_loader, normalizer,
    coords_orig, hilbert_idx, n_patches, wall_weights, data_stacked
    """
    # 1. Load data
    coords, pressure, u_vel, v_vel = auto_load_data(
        cfg, use_mock=use_mock, mock_coord_shift=mock_coord_shift
    )
    N, T = pressure.shape
    print(f"New dataset — nodes: {N:,}, timesteps: {T}")

    # 2. Temporal split
    T_train = max(int(T * cfg.train_ratio), cfg.time_in + cfg.time_out + 1)
    T_val   = max(int(T * cfg.val_ratio),   cfg.time_in + cfg.time_out + 1)
    T_test  = T - T_train - T_val
    print(f"Temporal split — train: [0, {T_train}), "
          f"val: [{T_train}, {T_train+T_val}), "
          f"test: [{T_train+T_val}, {T})")

    # 3. Normalize — using original coord bounds if provided ("fixed" mode)
    normalizer = DataNormalizer()
    tl_cfg = cfg if isinstance(cfg, TransferLearningConfig) else None

    if (
        tl_cfg is not None
        and tl_cfg.coord_alignment_mode == "fixed"
        and tl_cfg.original_coord_bounds is not None
    ):
        print("Using FIXED coordinate bounds from original training domain.")
        coords_norm = normalize_coords_to_training_domain(
            coords, tl_cfg.original_coord_bounds
        )
        # Fit variable scalers on training slice, but use fixed coord normalization
        normalizer.fit(
            pressure[:, :T_train],
            u_vel[:, :T_train],
            v_vel[:, :T_train],
            coords,
        )
        # Override coord transform to use our fixed normalization
        normalizer.coord_min   = np.array([
            tl_cfg.original_coord_bounds["x_min"],
            tl_cfg.original_coord_bounds["y_min"],
        ], dtype=np.float32)
        normalizer.coord_max   = np.array([
            tl_cfg.original_coord_bounds["x_max"],
            tl_cfg.original_coord_bounds["y_max"],
        ], dtype=np.float32)
        normalizer.coord_scale = normalizer.coord_max - normalizer.coord_min
        normalizer.coord_scale[normalizer.coord_scale < 1e-12] = 1.0
        p_n = normalizer.transform_var(pressure, "pressure")
        u_n = normalizer.transform_var(u_vel,    "u_velocity")
        v_n = normalizer.transform_var(v_vel,    "v_velocity")
        c_n = coords_norm
    else:
        print("Using ADAPTIVE coordinate normalization (new data bounds).")
        p_n, u_n, v_n, c_n = normalizer.fit_transform(
            pressure, u_vel, v_vel, coords, T_train
        )

    # 4. Hilbert sort
    print("Computing Hilbert sort …")
    hilbert_idx = hilbert_sort_indices(coords)

    # 5. Stack (N, T, C)
    x_rep = np.repeat(c_n[:, 0:1], T, axis=1)
    y_rep = np.repeat(c_n[:, 1:2], T, axis=1)
    data  = np.stack([p_n, u_n, v_n, x_rep, y_rep], axis=2).astype(np.float32)

    # 6. Wall weights
    wall_weights = None
    if cfg.use_spatial_weighting:
        n_patches_full = math.ceil(N / cfg.patch_size)
        wall_weights   = compute_wall_weights(c_n)

    # 7. Build datasets
    def _make_loader(t_start, t_end, shuffle):
        if t_end - t_start < cfg.time_in + cfg.time_out + 1:
            return DataLoader([], batch_size=cfg.batch_size)
        ds = PatchDataset(
            data, hilbert_idx, cfg.patch_size,
            cfg.time_in, cfg.time_out, t_start, t_end
        )
        if len(ds) == 0:
            return DataLoader([], batch_size=cfg.batch_size)
        return DataLoader(
            ds, batch_size=cfg.batch_size, shuffle=shuffle,
            num_workers=0, pin_memory=torch.cuda.is_available(),
        )

    train_loader = _make_loader(0,           T_train,          shuffle=True)
    val_loader   = _make_loader(T_train,     T_train + T_val,  shuffle=False)
    test_loader  = _make_loader(T_train + T_val, T,            shuffle=False)

    n_patches = math.ceil(N / cfg.patch_size)
    print(f"Patches: {n_patches}  |  "
          f"Train batches: {len(train_loader)}  |  "
          f"Val batches: {len(val_loader)}")

    return (
        train_loader, val_loader, test_loader,
        normalizer, coords, hilbert_idx, n_patches, wall_weights, data,
    )


print("build_dataloaders ready.")


## 18 — Summary Report

In [ ]:
def save_summary(
    cfg:     TransferLearningConfig,
    history: dict,
    output_dir: str,
) -> None:
    """Save a plain-text summary of the transfer learning run."""
    os.makedirs(output_dir, exist_ok=True)
    path = os.path.join(output_dir, "transfer_learning_summary.txt")

    best_val = min(
        (v for v in history["val_loss"] if not math.isnan(v)), default=float("nan")
    )
    epochs_ran = len(history["train_loss"])

    lines = [
        "=" * 60,
        "  ST-Mamba Transfer Learning Summary",
        "=" * 60,
        f"  Run timestamp        : {RUN_TIMESTAMP}",
        f"  Pre-trained ckpt     : {cfg.pretrained_checkpoint}",
        f"  Fine-tune LR         : {cfg.fine_tune_lr}",
        f"  Epochs (ran)         : {epochs_ran}",
        f"  Frozen blocks        : {cfg.freeze_n_blocks}",
        f"  Freeze embedding     : {cfg.freeze_embedding}",
        f"  Coord alignment mode : {cfg.coord_alignment_mode}",
        f"  Best val loss        : {best_val:.6f}",
        "",
        "  Final RMSE (physical units):",
    ]
    if history.get("val_rmse"):
        last_rmse = history["val_rmse"][-1]
        lines += [
            f"    Pressure   : {last_rmse[0]:.4f}",
            f"    U-velocity : {last_rmse[1]:.4f}",
            f"    V-velocity : {last_rmse[2]:.4f}",
        ]
    lines += ["=" * 60]

    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")
    print("\n".join(lines))
    print(f"\nSummary saved to {path}")


print("Summary report utility ready.")


## 19 — Run Transfer Learning Pipeline

This section ties everything together:

1. Load new dataset (or mock data)
2. Analyze coordinate shift vs training domain
3. Build dataloaders with appropriate coordinate normalization
4. Load pre-trained weights
5. Optionally freeze layers
6. Fine-tune with transfer learning loop
7. Save summary


In [ ]:
# ── Step 1: Load new dataset ──────────────────────────────────────────────────
print("\n[1/7] Loading new dataset …")
(train_loader, val_loader, test_loader,
 normalizer, coords, hilbert_idx, n_patches,
 wall_weights, data_stacked) = build_dataloaders(
    TL_CONFIG, use_mock=USE_MOCK, mock_coord_shift=(0.05, 0.02)
)

# ── Step 2: Coordinate shift analysis ────────────────────────────────────────
print("\n[2/7] Coordinate shift analysis …")
# For mock data we construct a reference; for real data supply old_coords
ref_coords = coords - np.array([0.05, 0.02], dtype=np.float32)  # mock "original"
shift, scale_ratio = analyze_coordinate_shift(ref_coords, coords)


In [ ]:
# ── Step 3: Create model ──────────────────────────────────────────────────────
print("\n[3/7] Creating model …")
model = create_model(TL_CONFIG, n_patches)

# ── Step 4: Load pre-trained weights ──────────────────────────────────────────
print("\n[4/7] Loading pre-trained weights …")
if TL_CONFIG.pretrained_checkpoint and os.path.exists(TL_CONFIG.pretrained_checkpoint):
    model = load_pretrained_weights(model, TL_CONFIG.pretrained_checkpoint, DEVICE)
else:
    print("  No pre-trained checkpoint provided — using random initialization.")
    print("  (Set PRETRAINED_CHECKPOINT to fine-tune a real model.)")

# ── Step 5: Freeze layers (optional) ─────────────────────────────────────────
print("\n[5/7] Configuring trainable layers …")
model = freeze_model_layers(
    model,
    n_blocks_to_freeze=TL_CONFIG.freeze_n_blocks,
    freeze_embedding=TL_CONFIG.freeze_embedding,
)


In [ ]:
# ── Step 6: Transfer learning training ────────────────────────────────────────
print("\n[6/7] Starting transfer learning training …")
criterion = STMambaLoss(TL_CONFIG, wall_weights=wall_weights,
                        patch_size=TL_CONFIG.patch_size).to(DEVICE)

history = train_transfer_learning(
    model, train_loader, val_loader, criterion,
    normalizer, TL_CONFIG, DEVICE,
)

# ── Step 7: Save summary and plots ────────────────────────────────────────────
print("\n[7/7] Saving summary and plots …")
plot_training_history(history, TL_CONFIG.output_dir)
save_summary(TL_CONFIG, history, TL_CONFIG.output_dir)
print(f"\n✔ Transfer learning complete.  Outputs in: {TL_CONFIG.output_dir}")


## 20 — Autoregressive Inference on Fine-Tuned Model

Load the best checkpoint from transfer learning and run multi-step rollout.


In [ ]:
# ── Load best TL checkpoint for inference ─────────────────────────────────────
best_tl_ckpt = os.path.join(TL_CONFIG.checkpoint_dir, "best_checkpoint.pth")
latest_tl_ckpt = os.path.join(TL_CONFIG.checkpoint_dir, "latest_checkpoint.pth")

inference_ckpt = best_tl_ckpt if os.path.exists(best_tl_ckpt) else latest_tl_ckpt

if os.path.exists(inference_ckpt):
    print(f"Loading fine-tuned weights from: {inference_ckpt}")
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    ckpt_data = torch.load(inference_ckpt, map_location=DEVICE)
    raw_model.load_state_dict(ckpt_data["model_state"])
    print("  ✔ Fine-tuned weights loaded for inference.")
else:
    print("No TL checkpoint found — using current model weights for inference.")


In [ ]:
# ── Prepare seed window from test split ───────────────────────────────────────
N, T, C = data_stacked.shape
T_train  = max(int(T * TL_CONFIG.train_ratio), TL_CONFIG.time_in + TL_CONFIG.time_out + 1)
T_val    = max(int(T * TL_CONFIG.val_ratio),   TL_CONFIG.time_in + TL_CONFIG.time_out + 1)
seed_t   = T_train + T_val   # first test timestep

if seed_t + TL_CONFIG.time_in >= T:
    seed_t = max(0, T - TL_CONFIG.time_in - 1)
    print(f"Dataset too short for test split — using seed_t={seed_t}")

initial_window, _ = prepare_seed_window(
    data_stacked, hilbert_idx, TL_CONFIG.patch_size, TL_CONFIG.time_in, seed_t
)
print(f"Seed window shape: {initial_window.shape}  (seed_t={seed_t})")

# ── Autoregressive rollout ────────────────────────────────────────────────────
print(f"\nRunning {TL_CONFIG.rollout_steps}-step autoregressive rollout …")
pred_rollout = autoregressive_rollout(
    model, initial_window, TL_CONFIG.rollout_steps, DEVICE, use_amp=TL_CONFIG.use_amp
)
print(f"Rollout shape: {pred_rollout.shape}  (steps, patches, patch_size, 3)")

# Reshape to (R, N_padded, 3) for export
R, P, S, _ = pred_rollout.shape
pred_flat   = pred_rollout.numpy().reshape(R, P * S, 3)
print(f"Predictions reshaped: {pred_flat.shape}")


## 21 — Export Predictions to CSV

In [ ]:
csv_paths = export_predictions_to_csv(
    pred_flat, coords, normalizer, TL_CONFIG.output_dir,
    prefix="st_mamba_tl_pred",
)

print("\nExported prediction CSVs:")
for p in csv_paths:
    print(f"  • {p}")


## 22 — Visualize Predictions

In [ ]:
plot_prediction_samples(
    pred_flat, coords, normalizer, TL_CONFIG.output_dir, n_cols=3
)


## 23 — Inspect Results

In [ ]:
print("=" * 60)
print("  Transfer Learning & Inference Summary")
print("=" * 60)
print(f"  Output directory     : {TL_CONFIG.output_dir}")
print(f"  Prediction shape     : {pred_flat.shape}  (steps, nodes, 3)")
print(f"  Coord array shape    : {coords.shape}")
print(f"  Rollout steps        : {TL_CONFIG.rollout_steps}")
print()

# Show physical range of predictions
var_names = ["Pressure", "U-velocity", "V-velocity"]
scalers   = ["pressure", "u_velocity", "v_velocity"]
for v_idx, (var, sc) in enumerate(zip(var_names, scalers)):
    N_nodes = min(len(coords), pred_flat.shape[1])
    raw  = pred_flat[:, :N_nodes, v_idx]
    phys = normalizer.inverse_transform_var(raw, sc)
    print(f"  {var:<14}: min={phys.min():.3f}  max={phys.max():.3f}  "
          f"mean={phys.mean():.3f}")

print()
print("Output files:")
for p in csv_paths:
    print(f"  • {p}")
